# Audio Augmentation for Google Colab — 5 Voice Types (WAV output)
อัปโหลดไฟล์เสียงนามสกุลใดก็ได้ → แปลงเป็น .wav (16,000 Hz) → สร้าง **1000 เสียง WAV** ที่แตกต่างกันต่อ 1 ไฟล์ → ดาวน์โหลด **เร็ว**

**🔧 รุ่นนี้แก้ปัญหา 'ไฟล์ไม่เป็น .wav'**
- เปลี่ยน default เป็น `OUTPUT_FORMAT = 'wav'` ทุกไฟล์ออกมาเป็น .wav จริง (PCM 16-bit, 16 kHz)
- แต่ยังเก็บเทคนิคทำให้ดาวน์โหลดเร็วไว้ทั้งหมด (Drive mount + ZIP_STORED + split)

## 1. ติดตั้ง Library

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q pydub librosa soundfile numpy scipy ipywidgets tqdm pyworld

## 2. ตั้งค่า — บังคับเป็น WAV

In [ ]:
# ============== ตัวเลือก ==============
USE_DRIVE = True              # บันทึก zip ลง Google Drive (ดาวน์โหลดเร็วสุด)
OUTPUT_FORMAT = 'wav'         # ⭐ 'wav' = ออกเป็น .wav เสมอ (เปลี่ยนกลับเป็น default)
FAST_ZIP = True               # True = zip แบบไม่บีบอัด (เร็วกว่าเดิม 10 เท่า)
SPLIT_ZIP_PARTS = 1           # 1 = ไม่แบ่ง, 5 = แบ่ง 5 ส่วน
# ======================================

import os, shutil, zipfile, random, time
import numpy as np
import librosa
import soundfile as sf
import pyworld as pw
from pydub import AudioSegment
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/audio_augmentation'
else:
    BASE_DIR = '/content/audio_project'

INPUT_DIR = os.path.join(BASE_DIR, 'input')
WAV_DIR = os.path.join(BASE_DIR, 'converted_wav')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
ZIP_DIR = os.path.join(BASE_DIR, 'zips')
for d in (INPUT_DIR, WAV_DIR, OUTPUT_DIR, ZIP_DIR):
    os.makedirs(d, exist_ok=True)

NUM_AUGMENTATIONS = 1000
TARGET_SR = 16000
FRAME_PERIOD = 5.0
MAX_RETRY_PASSES = 3
ZIP_MODE = zipfile.ZIP_STORED if FAST_ZIP else zipfile.ZIP_DEFLATED

# บังคับให้นามสกุลเป็น .wav และใช้ format='WAV' ของ soundfile
assert OUTPUT_FORMAT == 'wav', 'รุ่นนี้บังคับเป็น wav เท่านั้น'
EXT = 'wav'
SUBTYPE = 'PCM_16'
WRITE_FORMAT = 'WAV'

print(f'BASE_DIR     : {BASE_DIR}')
print(f'Output       : .{EXT} (PCM 16-bit, {TARGET_SR} Hz, mono)')
print(f'Zip mode     : {"STORED (ไว)" if FAST_ZIP else "DEFLATED (บีบ)"}')
print(f'Drive        : {USE_DRIVE}')
print(f'Target       : {NUM_AUGMENTATIONS} เสียง/ไฟล์')

## 3. อัปโหลดไฟล์เสียง

In [ ]:
for f in os.listdir(INPUT_DIR):
    os.remove(os.path.join(INPUT_DIR, f))

uploaded = files.upload()
for fname, data in uploaded.items():
    with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
        f.write(data)

print(f'\nอัปโหลด {len(uploaded)} ไฟล์:')
for f in sorted(os.listdir(INPUT_DIR)):
    print('  •', f)

## 4. แปลงไฟล์ต้นฉบับเป็น .wav (16 kHz, mono, 16-bit PCM)

In [ ]:
def convert_to_wav(src, dst, sr=TARGET_SR):
    try:
        a = AudioSegment.from_file(src)
        a = a.set_channels(1).set_frame_rate(sr).set_sample_width(2)
        a.export(dst, format='wav')
        return True
    except Exception as e:
        print(f'แปลงไม่สำเร็จ {src}: {e}')
        return False

for f in os.listdir(WAV_DIR):
    os.remove(os.path.join(WAV_DIR, f))
files_in = sorted(os.listdir(INPUT_DIR))
print(f'แปลง {len(files_in)} ไฟล์ ...')
wav_files = []
for fname in tqdm(files_in):
    src = os.path.join(INPUT_DIR, fname)
    stem = os.path.splitext(fname)[0]
    dst = os.path.join(WAV_DIR, f'{stem}.wav')
    if convert_to_wav(src, dst):
        wav_files.append(dst)
print(f'เสร็จ {len(wav_files)} ไฟล์ (.wav)')

## 5. WORLD vocoder

In [ ]:
def world_analyze_fast(y, sr, frame_period=FRAME_PERIOD):
    y = np.asarray(y, dtype=np.float64)
    f0, t = pw.dio(y, sr, frame_period=frame_period)
    f0 = pw.stonemask(y, f0, t, sr)
    sp = pw.cheaptrick(y, f0, t, sr)
    ap = pw.d4c(y, f0, t, sr)
    return f0, sp, ap

def warp_spectral_envelope_fast(sp, ratio):
    if abs(ratio - 1.0) < 1e-3:
        return sp
    n_frames, n_bins = sp.shape
    dst_idx = np.clip(np.arange(n_bins, dtype=np.float64) / ratio, 0, n_bins - 1)
    idx_floor = np.floor(dst_idx).astype(np.int32)
    idx_ceil = np.minimum(idx_floor + 1, n_bins - 1)
    frac = (dst_idx - idx_floor).astype(np.float64)
    out = sp[:, idx_floor] * (1.0 - frac) + sp[:, idx_ceil] * frac
    return np.maximum(out, 1e-16)

def voice_transform_precomputed(f0, sp, ap, sr, f0_ratio=1.0, formant_ratio=1.0,
                                jitter=0.0, shimmer=0.0, breathiness=0.0,
                                rng_np=None, frame_period=FRAME_PERIOD):
    if rng_np is None:
        rng_np = np.random.default_rng()
    f0_mod = (f0 * f0_ratio).astype(np.float64)
    if jitter > 0:
        voiced = f0_mod > 0
        noise = rng_np.normal(1.0, jitter, size=f0_mod.shape)
        f0_mod = np.where(voiced, f0_mod * noise, f0_mod)
        f0_mod = np.maximum(f0_mod, 0)
    sp_mod = warp_spectral_envelope_fast(sp, formant_ratio).astype(np.float64) if abs(formant_ratio - 1.0) > 1e-3 else sp
    ap_mod = np.clip(ap + breathiness, 0, 1).astype(np.float64) if breathiness > 0 else ap
    f0_mod = np.ascontiguousarray(f0_mod)
    sp_mod = np.ascontiguousarray(sp_mod)
    ap_mod = np.ascontiguousarray(ap_mod)
    y_out = pw.synthesize(f0_mod, sp_mod, ap_mod, sr, frame_period)
    if shimmer > 0:
        t = np.arange(len(y_out)) / sr
        rate_hz = rng_np.uniform(4, 8)
        mod = 1.0 - shimmer * (0.5 + 0.5 * np.sin(2 * np.pi * rate_hz * t))
        y_out = y_out * mod
    return y_out.astype(np.float32)

def semitones_to_ratio(st):
    return 2 ** (st / 12.0)

print('WORLD vocoder พร้อม')

## 6. Voice Presets — 5 ประเภท + ฟังก์ชันเขียน .wav

In [ ]:
def voice_child(f0, sp, ap, sr, rng, rng_np):
    return voice_transform_precomputed(f0, sp, ap, sr,
        f0_ratio=semitones_to_ratio(rng.uniform(5, 9)),
        formant_ratio=rng.uniform(1.15, 1.35), rng_np=rng_np), 'child'

def voice_female(f0, sp, ap, sr, rng, rng_np):
    return voice_transform_precomputed(f0, sp, ap, sr,
        f0_ratio=semitones_to_ratio(rng.uniform(2, 5)),
        formant_ratio=rng.uniform(1.06, 1.18), rng_np=rng_np), 'female'

def voice_male(f0, sp, ap, sr, rng, rng_np):
    return voice_transform_precomputed(f0, sp, ap, sr,
        f0_ratio=semitones_to_ratio(-rng.uniform(2, 5)),
        formant_ratio=rng.uniform(0.85, 0.96), rng_np=rng_np), 'male'

def voice_elderly_female(f0, sp, ap, sr, rng, rng_np):
    return voice_transform_precomputed(f0, sp, ap, sr,
        f0_ratio=semitones_to_ratio(rng.uniform(1, 3.5)),
        formant_ratio=rng.uniform(1.04, 1.14),
        jitter=rng.uniform(0.025, 0.06),
        shimmer=rng.uniform(0.08, 0.18),
        breathiness=rng.uniform(0.06, 0.16),
        rng_np=rng_np), 'elderly_female'

def voice_elderly_male(f0, sp, ap, sr, rng, rng_np):
    return voice_transform_precomputed(f0, sp, ap, sr,
        f0_ratio=semitones_to_ratio(-rng.uniform(1.5, 4.5)),
        formant_ratio=rng.uniform(0.88, 0.97),
        jitter=rng.uniform(0.025, 0.06),
        shimmer=rng.uniform(0.08, 0.18),
        breathiness=rng.uniform(0.06, 0.16),
        rng_np=rng_np), 'elderly_male'

VOICE_PRESETS = [
    (voice_child,          200),
    (voice_female,         200),
    (voice_male,           200),
    (voice_elderly_female, 200),
    (voice_elderly_male,   200),
]
assert sum(c for _, c in VOICE_PRESETS) == NUM_AUGMENTATIONS

def post_random_tweak(y, sr, rng, rng_np):
    if rng.random() < 0.7:
        y = y * (10 ** (rng.uniform(-3, 3) / 20))
    if rng.random() < 0.3:
        y = y + rng_np.normal(0, rng.uniform(0.0005, 0.003), len(y)).astype(np.float32)
    peak = np.max(np.abs(y))
    if peak > 1.0:
        y = y / peak * 0.99
    return y.astype(np.float32)

def write_wav(path, y, sr):
    """บังคับเขียนเป็น .wav PCM 16-bit จริง ๆ"""
    sf.write(path, y, sr, subtype='PCM_16', format='WAV')

def synthesize_one(fn, i, stem, out_folder, sr, f0, sp, ap):
    seed = hash((stem, i, fn.__name__)) & 0xFFFFFFFF
    rng = random.Random(seed)
    rng_np = np.random.default_rng(seed)
    try:
        y_out, label = fn(f0, sp, ap, sr, rng, rng_np)
        y_out = post_random_tweak(y_out, sr, rng, rng_np)
        if len(y_out) == 0 or not np.isfinite(y_out).all():
            return False, 'empty/nan output'
        out_path = os.path.join(out_folder, f'{stem}_{i:04d}_{label}.wav')
        write_wav(out_path, y_out, sr)
        return True, None
    except Exception as e:
        return False, str(e)

def augment_one_file(wav_path, out_folder, sr=TARGET_SR):
    os.makedirs(out_folder, exist_ok=True)
    for f in os.listdir(out_folder):
        os.remove(os.path.join(out_folder, f))
    audio, _ = librosa.load(wav_path, sr=sr, mono=True)
    audio = audio.astype(np.float32)
    stem = os.path.splitext(os.path.basename(wav_path))[0]
    write_wav(os.path.join(out_folder, f'{stem}_0000_original.wav'), audio, sr)
    print(f'  ▸ analyze (1 ครั้ง) ...')
    f0, sp, ap = world_analyze_fast(audio, sr)
    tasks = []
    counter = 1
    for fn, count in VOICE_PRESETS:
        for _ in range(count):
            tasks.append((fn, counter))
            counter += 1
    print(f'  ▸ synthesize {len(tasks)} เสียง (.wav) ...')
    failed = []
    for fn, i in tqdm(tasks, desc=f'  {stem}', leave=False):
        ok, err = synthesize_one(fn, i, stem, out_folder, sr, f0, sp, ap)
        if not ok:
            failed.append((fn, i, err))
    retry = 0
    while failed and retry < MAX_RETRY_PASSES:
        retry += 1
        print(f'  ▸ retry pass #{retry}: {len(failed)} ไฟล์ ...')
        new_failed = []
        for fn, i, _ in failed:
            ok, err = synthesize_one(fn, i, stem, out_folder, sr, f0, sp, ap)
            if not ok:
                new_failed.append((fn, i, err))
        failed = new_failed
    files_in_folder = [f for f in os.listdir(out_folder) if f.endswith('.wav')]
    augmented_count = len(files_in_folder) - 1
    if augmented_count >= NUM_AUGMENTATIONS:
        print(f'  ✓ ครบ {NUM_AUGMENTATIONS} เสียง (.wav)')
    else:
        print(f'  ⚠ ขาด {NUM_AUGMENTATIONS - augmented_count} ไฟล์ — ตัวอย่าง: {failed[:3]}')
    return out_folder, augmented_count

print(f'พร้อม — สัดส่วน {NUM_AUGMENTATIONS} เสียง (.wav):')
for fn, c in VOICE_PRESETS:
    print(f'  {fn.__name__:22s} : {c} ไฟล์')

## 7. ฟัง preview แต่ละประเภท

In [ ]:
from IPython.display import Audio
wav_list = sorted([os.path.join(WAV_DIR, f) for f in os.listdir(WAV_DIR) if f.endswith('.wav')])
if wav_list:
    sample, _ = librosa.load(wav_list[0], sr=TARGET_SR, mono=True)
    sample = sample.astype(np.float32)
    print('ตัวอย่าง:', os.path.basename(wav_list[0]))
    display(Audio(sample, rate=TARGET_SR))
    f0, sp, ap = world_analyze_fast(sample, TARGET_SR)
    for fn, _ in VOICE_PRESETS:
        rng = random.Random(42); rng_np = np.random.default_rng(42)
        try:
            y2, label = fn(f0, sp, ap, TARGET_SR, rng, rng_np)
            print(f'\n=== {label} ===')
            display(Audio(y2, rate=TARGET_SR))
        except Exception as e:
            print(f'{fn.__name__} error: {e}')

## 8. ฟังก์ชัน zip + ดาวน์โหลด

In [ ]:
def fast_zip_folder(folder_path, zip_path, mode=ZIP_MODE):
    with zipfile.ZipFile(zip_path, 'w', mode, allowZip64=True) as zf:
        for root, _, fs in os.walk(folder_path):
            for f in fs:
                full = os.path.join(root, f)
                arc = os.path.relpath(full, os.path.dirname(folder_path))
                zf.write(full, arc)
    return zip_path

def split_zip_files(folder_path, zip_path_template, n_parts):
    all_files = []
    for root, _, fs in os.walk(folder_path):
        for f in fs:
            all_files.append(os.path.join(root, f))
    all_files.sort()
    chunk_size = (len(all_files) + n_parts - 1) // n_parts
    out_paths = []
    for i in range(n_parts):
        chunk = all_files[i*chunk_size:(i+1)*chunk_size]
        if not chunk:
            continue
        zp = zip_path_template.format(part=i+1, total=n_parts)
        with zipfile.ZipFile(zp, 'w', ZIP_MODE, allowZip64=True) as zf:
            for full in chunk:
                arc = os.path.relpath(full, os.path.dirname(folder_path))
                zf.write(full, arc)
        out_paths.append(zp)
    return out_paths

def make_drive_link(file_path):
    if not file_path.startswith('/content/drive/'):
        return None
    rel = file_path.replace('/content/drive/MyDrive/', '')
    drive_url = 'https://drive.google.com/drive/my-drive'
    size_mb = os.path.getsize(file_path) / 1024 / 1024
    html = f'''
    <div style="padding:10px;background:#e8f5e9;border-left:4px solid #4caf50;margin:6px 0;">
        <b>📁 อยู่ใน Drive ({size_mb:.1f} MB):</b><br>
        <code>MyDrive/{rel}</code><br>
        <a href="{drive_url}" target="_blank" style="display:inline-block;background:#4caf50;color:white;padding:6px 14px;border-radius:4px;text-decoration:none;margin-top:6px;">เปิด Google Drive →</a>
    </div>'''
    display(HTML(html))

def make_download_button(zip_path, label):
    btn = widgets.Button(description=f'⬇ ดาวน์โหลด {label}',
                         button_style='success',
                         layout=widgets.Layout(width='500px'))
    out = widgets.Output()
    def _on_click(_):
        with out:
            clear_output()
            print(f'กำลังส่ง {os.path.basename(zip_path)} ...')
            files.download(zip_path)
    btn.on_click(_on_click)
    display(widgets.VBox([btn, out]))

print('ฟังก์ชัน zip + download พร้อม')

## 9. รันประมวลผลทุกไฟล์ — output เป็น .wav ทุกไฟล์

In [ ]:
wav_list = sorted([os.path.join(WAV_DIR, f) for f in os.listdir(WAV_DIR) if f.endswith('.wav')])
print(f'พบ wav {len(wav_list)} ไฟล์ — สร้าง {NUM_AUGMENTATIONS} เสียง (.wav) / ไฟล์\n')

summary = []
t_total = time.time()
for idx, wav_path in enumerate(wav_list, 1):
    stem = os.path.splitext(os.path.basename(wav_path))[0]
    folder = os.path.join(OUTPUT_DIR, stem)
    print(f'\n[{idx}/{len(wav_list)}] ▶ {stem}')
    t0 = time.time()
    _, n_aug = augment_one_file(wav_path, folder)
    elapsed_synth = time.time() - t0
    print(f'  ▸ zip ...')
    t_zip = time.time()
    if SPLIT_ZIP_PARTS > 1:
        template = os.path.join(ZIP_DIR, stem + '_part{part}of{total}.zip')
        zip_paths = split_zip_files(folder, template, SPLIT_ZIP_PARTS)
    else:
        zp = os.path.join(ZIP_DIR, f'{stem}.zip')
        fast_zip_folder(folder, zp)
        zip_paths = [zp]
    elapsed_zip = time.time() - t_zip
    total_size = sum(os.path.getsize(z) for z in zip_paths) / 1024 / 1024
    status = '✓' if n_aug == NUM_AUGMENTATIONS else f'⚠ ขาด {NUM_AUGMENTATIONS - n_aug}'
    print(f'  {status} synth {elapsed_synth:.1f}s | zip {elapsed_zip:.1f}s | {total_size:.1f} MB')
    summary.append((stem, n_aug, elapsed_synth, total_size))
    for zp in zip_paths:
        if USE_DRIVE:
            make_drive_link(zp)
        size = os.path.getsize(zp) / 1024 / 1024
        make_download_button(zp, f'{os.path.basename(zp)} ({size:.1f} MB)')

print(f'\n=== สรุป (ใช้เวลารวม {(time.time()-t_total)/60:.1f} นาที) ===')
for stem, n, t, mb in summary:
    mark = '✓' if n == NUM_AUGMENTATIONS else '⚠'
    print(f'  {mark} {stem:30s} : {n} เสียง — {t:.1f}s — {mb:.1f} MB')

## 10. ตรวจสอบว่าไฟล์เป็น .wav จริง

In [ ]:
# สุ่มเปิดไฟล์ output ขึ้นมาตรวจสอบ header
for sub in sorted(os.listdir(OUTPUT_DIR))[:3]:
    p = os.path.join(OUTPUT_DIR, sub)
    if not os.path.isdir(p):
        continue
    wavs = sorted([f for f in os.listdir(p) if f.endswith('.wav')])
    if not wavs:
        continue
    sample_file = os.path.join(p, wavs[len(wavs)//2])
    info = sf.info(sample_file)
    print(f'{os.path.basename(sample_file)}')
    print(f'   format       = {info.format}     (ต้องเป็น WAV)')
    print(f'   subtype      = {info.subtype}    (ต้องเป็น PCM_16)')
    print(f'   sample_rate  = {info.samplerate} Hz')
    print(f'   channels     = {info.channels}')
    print(f'   duration     = {info.duration:.2f} s\n')

print('จำนวนไฟล์ .wav ในแต่ละโฟลเดอร์:')
for sub in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, sub)
    if os.path.isdir(p):
        files_count = len([f for f in os.listdir(p) if f.endswith('.wav')])
        mark = '✓' if files_count == NUM_AUGMENTATIONS + 1 else '⚠'
        print(f'  {mark} {sub:30s} : {files_count} ไฟล์ (เป้า: {NUM_AUGMENTATIONS+1})')